# HeatUp — Production Notebook 2: Stability Assessment

Runs the HeatUp three-gate stability pipeline for all validated materials in the database.

| Gate | Criterion | Cost |
|------|-----------|------|
| 1 — Mechanical | Born–Huang: all C eigenvalues > 0, B > threshold, G > threshold | ~0 s (reads elastic_tensor.json) |
| 2 — Vibrational | VDOS soft-mode fraction ζ < fail threshold | ~0 s (reads cached vdos.json) |
| 3 — Thermodynamic | E_above_hull(T_op) < threshold via T-dependent convex hull | ~min (reads/generates competing phases) |

**Prerequisites** (produced by notebook 1):
- `elastic/elastic_tensor.json` — Gate 1
- `aimd/<T>K/anharmonic_phonons/vdos.json` — Gate 2
- `relaxation/energy.json` + `phonons/dos.json` — Gate 3 (target + competing phases)

**Outputs per material:**
- `stability/stability_report.json` — machine-readable verdict
- `stability/stability_report.pdf` — three-panel dashboard
- `stability/hull_vs_T.json` — full hull data
- `stability/secondary_phases.json` — list of competing phases used

---

## 0 — Imports

In [ ]:
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image

# ── HeatUp stability framework ────────────────────────────────────────────────
HEATUP_ROOT = os.path.abspath(os.path.join('..', '..'))
if HEATUP_ROOT not in sys.path:
    sys.path.insert(0, HEATUP_ROOT)

from heatup import (
    run_stability_pipeline,
    run_stability_pipeline_batch,
    assess_mechanical_stability,
    assess_vibrational_stability,
)
import heatup.config as cfg

# ── SSE-AL library ────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from library.md_pipeline import scan_database, print_database_summary

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'HeatUp config thresholds:')
print(f'  Born eigenvalue fail : {cfg.MECH_BORN_EIGENVALUE_FAIL_GPa} GPa')
print(f'  Bulk modulus warn    : {cfg.MECH_BULK_WARN_GPa} GPa')
print(f'  Shear modulus warn   : {cfg.MECH_SHEAR_WARN_GPa} GPa')
print(f'  VDOS zero-mode warn  : {cfg.VIB_ZERO_FRAC_WARN * 100:.0f}%')
print(f'  VDOS zero-mode fail  : {cfg.VIB_ZERO_FRAC_FAIL * 100:.0f}%')
print(f'  VDOS min frames      : {cfg.VIB_MIN_FRAMES}')
print(f'  Hull warn threshold  : {cfg.THERMO_HULL_WARN_EV * 1000:.0f} meV/atom')

## 1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATABASE_ROOT  = 'database'
CANDIDATES_DIR = 'input/candidates'

# ── Operating temperature for stability assessment ────────────────────────────
# Materials are classified stable/metastable/unstable at this temperature.
OPERATING_T = 1200.0    # K

# ── Hull temperature grid ─────────────────────────────────────────────────────
HULL_TEMPERATURES = list(range(0, 1501, 50))    # K

# ── Run mode ──────────────────────────────────────────────────────────────────
# 'batch'  — run all validated entries in DATABASE_ROOT
# 'manual' — run only the materials in MANUAL_LIST below
RUN_MODE = 'batch'

MANUAL_LIST = [
    ('AgI',      'P6_3mc'),
    ('Li2ZrCl6', 'R-3m'),
]

# ── Pipeline options ──────────────────────────────────────────────────────────
# Generate missing secondary phases with PyXtal before hull construction.
GENERATE_MISSING_PHASES = True

# Recompute even if stability_report.json already exists.
FORCE_RERUN = False

# Save three-panel dashboard figure.
SAVE_FIGURES = True

# ── Threshold overrides (optional) ───────────────────────────────────────────
# Uncomment to change from defaults:
# cfg.THERMO_HULL_WARN_EV      = 0.05   # 50 meV/atom (tighter)
# cfg.VIB_ZERO_FRAC_FAIL       = 0.10   # 10% (more lenient)
# cfg.MECH_BULK_WARN_GPa       = 20.0   # demand stiffer electrolytes

print('Configuration loaded.')
print(f'Operating temperature: {OPERATING_T:.0f} K')
print(f'Run mode: {RUN_MODE}')
print(f'Generate missing phases: {GENERATE_MISSING_PHASES}')

## 2 — Database state before assessment

In [ ]:
print_database_summary(DATABASE_ROOT)

records = scan_database(DATABASE_ROOT)
print(f'\nTotal completed simulations: {len(records)}')

# Show which materials have all prerequisites for stability assessment
print('\nPrerequisite check:')
print(f'{"Material/Symmetry":<35} {"Elastic":>8} {"VDOS":>6} {"E0":>4} {"DOS":>5}')
print('-' * 60)

seen = set()
for rec in records:
    mat = rec['material']
    sym = rec['symmetry']
    key = f'{mat}/{sym}'
    if key in seen:
        continue
    seen.add(key)

    sym_dir = os.path.join(DATABASE_ROOT, mat, sym)
    has_elastic = os.path.exists(os.path.join(sym_dir, 'elastic', 'elastic_tensor.json'))
    has_e0      = os.path.exists(os.path.join(sym_dir, 'relaxation', 'energy.json'))
    has_dos     = os.path.exists(os.path.join(sym_dir, 'phonons', 'dos.json'))

    # Check for any cached VDOS
    aimd_dir = os.path.join(sym_dir, 'aimd')
    has_vdos = False
    if os.path.isdir(aimd_dir):
        for tf in os.listdir(aimd_dir):
            vp = os.path.join(aimd_dir, tf, 'anharmonic_phonons', 'vdos.json')
            if os.path.exists(vp):
                has_vdos = True
                break

    e = lambda b: '✓' if b else '✗'
    print(f'{key:<35} {e(has_elastic):>8} {e(has_vdos):>6} {e(has_e0):>4} {e(has_dos):>5}')

## 3 — Run stability pipeline

In [ ]:
all_reports = []

if RUN_MODE == 'batch':
    # Run on every validated entry in the database
    all_reports = run_stability_pipeline_batch(
        database_root           = DATABASE_ROOT,
        candidates_root         = CANDIDATES_DIR,
        operating_T             = OPERATING_T,
        temperatures            = HULL_TEMPERATURES,
        device                  = DEVICE,
        generate_missing_phases = GENERATE_MISSING_PHASES,
        force_rerun             = FORCE_RERUN,
        save_figures            = SAVE_FIGURES,
    )

else:  # manual
    for mat, sym in MANUAL_LIST:
        sym_dir = os.path.join(DATABASE_ROOT, mat, sym)
        if not os.path.isdir(sym_dir):
            print(f'[skip] {mat}/{sym}: not found in database')
            continue
        report = run_stability_pipeline(
            sym_dir                 = sym_dir,
            operating_T             = OPERATING_T,
            candidates_root         = CANDIDATES_DIR,
            database_root           = DATABASE_ROOT,
            temperatures            = HULL_TEMPERATURES,
            device                  = DEVICE,
            generate_missing_phases = GENERATE_MISSING_PHASES,
            force_rerun             = FORCE_RERUN,
            save_figure             = SAVE_FIGURES,
        )
        all_reports.append(report)

print(f'\n{"=" * 55}')
print(f'Assessment complete: {len(all_reports)} materials')
n_ok   = sum(1 for r in all_reports if r['overall'] == cfg.STATUS_OK)
n_warn = sum(1 for r in all_reports if r['overall'] == cfg.STATUS_WARN)
n_fail = sum(1 for r in all_reports if r['overall'] == cfg.STATUS_FAIL)
n_miss = sum(1 for r in all_reports if r['overall'] == cfg.STATUS_MISSING)
print(f'  OK:      {n_ok}')
print(f'  WARN:    {n_warn}')
print(f'  FAIL:    {n_fail}')
print(f'  MISSING: {n_miss}')
print(f'{"=" * 55}')

## 4 — Results summary table

In [ ]:
if not all_reports:
    print('No reports to display.')
else:
    rows = []
    for r in all_reports:
        mech  = r.get('mechanical', {})
        vib   = r.get('vibrational', {})
        therm = r.get('thermodynamic', {})

        rows.append({
            'Material'       : r.get('material', '?'),
            'Symmetry'       : r.get('symmetry', '?'),
            'Mech'           : mech.get('status', '-'),
            'B (GPa)'        : f"{mech.get('bulk_modulus_GPa', float('nan')):.0f}" if mech.get('available') else '-',
            'G (GPa)'        : f"{mech.get('shear_modulus_GPa', float('nan')):.0f}" if mech.get('available') else '-',
            'Vib'            : vib.get('status', '-'),
            'ζ (%)'          : f"{vib.get('zero_window_frac', float('nan')) * 100:.1f}" if vib.get('available') else '-',
            'Thermo'         : therm.get('status', '-'),
            'ΔE_hull (meV)'  : f"{therm.get('e_above_hull_at_T_eV', float('nan')) * 1000:.1f}" if therm.get('available') else '-',
            'Stopped at'     : r.get('stopped_at', 'completed') or 'completed',
            'Overall'        : r.get('overall', '-'),
        })

    df = pd.DataFrame(rows)

    def _colour(val):
        colours = {
            'ok'     : 'background-color: #d5f5e3',
            'warn'   : 'background-color: #fef9e7',
            'fail'   : 'background-color: #fadbd8',
            'missing': 'background-color: #f2f2f2',
        }
        return colours.get(str(val).lower(), '')

    styled = df.style\
        .applymap(_colour, subset=['Mech', 'Vib', 'Thermo', 'Overall'])\
        .set_caption(f'Stability assessment at T_op = {OPERATING_T:.0f} K')
    display(styled)

## 5 — Detailed report for a single material

In [ ]:
INSPECT_MATERIAL = 'AgI'
INSPECT_SYMMETRY = 'P6_3mc'

# Find the report for this material
target_report = next(
    (r for r in all_reports
     if r.get('material') == INSPECT_MATERIAL and r.get('symmetry') == INSPECT_SYMMETRY),
    None
)

if target_report is None:
    print(f'{INSPECT_MATERIAL}/{INSPECT_SYMMETRY} not found in reports.')
    print(f'Available: {[(r["material"], r["symmetry"]) for r in all_reports]}')
else:
    print(f'=== {INSPECT_MATERIAL}/{INSPECT_SYMMETRY} ===')
    print(f'Overall: {target_report["overall"].upper()}')
    print()

    m = target_report.get('mechanical', {})
    if m.get('available'):
        print('Gate 1 — Mechanical:')
        print(f'  Status      : {m["status"]}')
        print(f'  Born stable : {m["born_stable"]}')
        print(f'  Min C eig   : {m["min_eigenvalue_GPa"]:.1f} GPa')
        print(f'  B           : {m["bulk_modulus_GPa"]:.1f} GPa')
        print(f'  G           : {m["shear_modulus_GPa"]:.1f} GPa')
        print()

    v = target_report.get('vibrational', {})
    if v.get('available'):
        print('Gate 2 — Vibrational:')
        print(f'  Status     : {v["status"]}')
        print(f'  ζ (|ω|<1meV): {v["zero_window_frac"] * 100:.2f}%')
        print(f'  N sources  : {v["n_sources"]}')
        print()

    t = target_report.get('thermodynamic', {})
    if t.get('available'):
        print('Gate 3 — Thermodynamic:')
        print(f'  Status           : {t["status"]}')
        print(f'  ΔE_hull ({OPERATING_T:.0f} K)  : {t["e_above_hull_at_T_eV"] * 1000:.1f} meV/atom')
        print(f'  Competing phases : {t["n_competing"]}')
        print(f'  Generated phases : {t.get("n_generated", 0)}')

    for flag in target_report.get('flags', []):
        print(f'  ⚠ {flag}')

    # Show dashboard figure
    sym_dir  = os.path.join(DATABASE_ROOT, INSPECT_MATERIAL, INSPECT_SYMMETRY)
    fig_path = os.path.join(sym_dir, 'stability', 'stability_report.pdf')
    png_path = fig_path.replace('.pdf', '.png')
    if os.path.exists(png_path):
        display(Image(png_path, width=900))
    elif os.path.exists(fig_path):
        print(f'Figure available: {fig_path} (open in PDF viewer)')

## 6 — Hull vs temperature plot

In [ ]:
# Plot ΔE_hull(T) for all stable/metastable materials
fig, ax = plt.subplots(figsize=(10, 5))

colours = plt.cm.tab20.colors
n_plotted = 0

for i, report in enumerate(all_reports):
    therm = report.get('thermodynamic', {})
    if not therm.get('available'):
        continue
    hull_results = therm.get('hull_results', [])
    if not hull_results:
        continue

    valid = [(r['T'], r['e_above_hull_eV_atom'])
             for r in hull_results
             if r.get('e_above_hull_eV_atom') is not None]
    if not valid:
        continue

    T_arr = np.array([v[0] for v in valid])
    E_meV = np.array([v[1] for v in valid]) * 1000
    label = f"{report['material']}/{report['symmetry']}"
    col   = colours[n_plotted % len(colours)]

    ax.plot(T_arr, E_meV, color=col, lw=1.2, label=label)
    n_plotted += 1

ax.axhline(0,                              color='green',  lw=1.0, ls='--', alpha=0.7, label='Stable limit')
ax.axhline(cfg.THERMO_HULL_WARN_EV * 1000, color='orange', lw=1.0, ls='--', alpha=0.7, label=f'Metastable limit ({cfg.THERMO_HULL_WARN_EV*1000:.0f} meV/atom)')
ax.axvline(OPERATING_T,                    color='gray',   lw=0.8, ls=':',  alpha=0.6, label=f'T_op = {OPERATING_T:.0f} K')

ax.set_xlabel('Temperature (K)', fontsize=11)
ax.set_ylabel('ΔE above hull (meV/atom)', fontsize=11)
ax.set_title(f'Temperature-dependent hull distances  (HeatUp, T_op = {OPERATING_T:.0f} K)', fontsize=11)
ax.set_xlim(0, max(HULL_TEMPERATURES))

if n_plotted <= 15:
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
else:
    ax.text(0.02, 0.98, f'{n_plotted} materials', transform=ax.transAxes,
            ha='left', va='top', fontsize=8)

plt.tight_layout()
plt.savefig('hull_vs_temperature.pdf', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: hull_vs_temperature.pdf')

## 7 — Export results for retraining

Writes `stability_results.json` with all reports for consumption by notebook 3
(active-learning retraining) and any downstream analysis.

In [ ]:
# Slim down reports: remove large VDOS arrays (stored in anharmonic_phonons/)
slim_reports = []
for r in all_reports:
    slim = {k: v for k, v in r.items() if k != 'vibrational'}
    slim['vibrational'] = {k: v for k, v in r.get('vibrational', {}).items()
                           if k not in ('omega_mev', 'vdos')}
    slim_reports.append(slim)

output_path = 'stability_results.json'
with open(output_path, 'w') as fh:
    json.dump(slim_reports, fh, indent=2)

print(f'Saved {len(slim_reports)} reports → {output_path}')

# Summary for retraining decision
stable_mats = [
    f"{r['material']}/{r['symmetry']}"
    for r in all_reports
    if r.get('overall') in (cfg.STATUS_OK, cfg.STATUS_WARN)
]
print(f'\nMaterials to include in retraining pool ({len(stable_mats)}):')
for m in stable_mats[:20]:
    print(f'  {m}')
if len(stable_mats) > 20:
    print(f'  ... and {len(stable_mats) - 20} more')

---
**Next step:** Run `03_anharmonic_analysis.ipynb` for detailed free-energy decomposition and VDOS visualisation.